# Stage 2: Instruction Fine-Tuning (SFT)
**Goal:** Teach the model to answer finance questions using 100 Q&A pairs.

## Step 1 — Install dependencies

In [1]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## Step 2 — Imports

In [2]:
import torch
import json
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

print(f"GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 3 — Configuration

In [5]:
from dataclasses import dataclass, asdict

@dataclass
class ModelArguments:
    MODEL_PATH   : str   = "/content/drive/MyDrive/finance_ft/stage1_adapter"
    MAX_SEQ_LEN  : int   = 512
    LOAD_IN_4BIT : bool  = True
    LORA_R       : int   = 16
    LORA_ALPHA   : int   = 16
    LORA_DROPOUT : float = 0.05
    TARGET_MODS  : list  = None
    OUTPUT_DIR   : str   = "./outputs/stage2_sft"
    EPOCHS       : int   = 3
    BATCH_SIZE   : int   = 2
    GRAD_ACCUM   : int   = 4
    LR           : float = 2e-4

    def __post_init__(self):
        if self.TARGET_MODS is None:
            self.TARGET_MODS = ["q_proj", "k_proj", "v_proj", "o_proj",
                               "up_proj", "down_proj", "gate_proj"]

# Create instance
args = ModelArguments()

# Access values like this
print(args.MAX_SEQ_LEN)   # 512
print(args.LORA_R)        # 16

512
16


In [6]:

# Alpaca prompt template
ALPACA_TEMPLATE = """Below is a finance question. Write a clear, accurate, and helpful response.

### Question:
{}

### Answer:
{}"""

ALPACA_TEMPLATE_INFERENCE = """Below is a finance question. Write a clear, accurate, and helpful response.

### Question:
{}

### Answer:
"""

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 4 — Load model
> Option A: Load from base model
> Option B: Load from Stage 1 adapter (recommended)

In [8]:
# OPTION A — Fresh base model
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name     = MODEL_NAME,
#     max_seq_length = MAX_SEQ_LEN,
#     dtype          = None,
#     load_in_4bit   = LOAD_IN_4BIT,
# )

# OPTION B — Continue from Stage 1 adapter (uncomment if using)
# Create instance first (do this once after the class definition)
args = ModelArguments()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = args.MODEL_PATH,
    max_seq_length = args.MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = args.LOAD_IN_4BIT,
)

print("Model loaded")

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.7.2 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Model loaded


## Step 5 — Apply LoRA

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = args.LORA_R,
    target_modules             = args.TARGET_MODS,
    lora_alpha                 = args.LORA_ALPHA,
    lora_dropout               = args.LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)
model.print_trainable_parameters()

Unsloth: Already have LoRA adapters! We shall skip this step.


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


## Step 6 — Load and format instruction dataset

In [10]:
from datasets import load_dataset

EOS = tokenizer.eos_token

# Load directly from HuggingFace
ds = load_dataset("gbharti/finance-alpaca")
print(ds)

# Filter and take 100 examples for instruction fine-tuning
records = [
    item for item in ds["train"]
    if len(item["instruction"].strip()) > 10
    and len(item["output"].strip()) > 50
][:100]

print(f"Total examples loaded: {len(records)}")
print(f"\nSample instruction: {records[0]['instruction']}")
print(f"Sample output:      {records[0]['output'][:150]}")

# Format into Alpaca template
def format_example(rec):
    return ALPACA_TEMPLATE.format(
        rec["instruction"].strip(),
        rec["output"].strip()
    ) + EOS

formatted = [format_example(r) for r in records]
dataset = Dataset.from_dict({"text": formatted})
print(f"\nDataset size: {len(dataset)}")
print(f"\nSample formatted example:")
print(formatted[0])

README.md:   0%|          | 0.00/831 [00:00<?, ?B/s]

Cleaned_date.json:   0%|          | 0.00/42.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/68912 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 68912
    })
})
Total examples loaded: 100

Sample instruction: For a car, what scams can be plotted with 0% financing vs rebate?
Sample output:      The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they 

Dataset size: 100

Sample formatted example:
Below is a finance question. Write a clear, accurate, and helpful response.

### Question:
For a car, what scams can be plotted with 0% financing vs rebate?

### Answer:
The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. 

In [12]:
# Filter out examples that exceed 506 tokens
filtered = []
skipped  = 0

for rec in records:
    formatted = format_example(rec)
    toks = tokenizer(
        formatted,
        return_tensors="pt"
    )["input_ids"].shape[1]

    if toks <= 506:
        filtered.append(formatted)
    else:
        skipped += 1

print(f"Total records    : {len(records)}")
print(f"Skipped too long : {skipped}")
print(f"kept for training: {len(filtered)}")

dataset = Dataset.from_dict({"text": filtered})
print(dataset)

Total records    : 100
Skipped too long : 15
kept for training: 85
Dataset({
    features: ['text'],
    num_rows: 85
})


## Step 7 — Train

In [16]:
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = args.MAX_SEQ_LEN,
    args = TrainingArguments(
        output_dir                  = args.OUTPUT_DIR,
        num_train_epochs            = args.EPOCHS,
        per_device_train_batch_size = args.BATCH_SIZE,
        gradient_accumulation_steps = args.GRAD_ACCUM,
        learning_rate               = args.LR,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        save_strategy               = "no",
        warmup_ratio                = 0.1,
        lr_scheduler_type           = "cosine",
        report_to                   = "none",
    ),
)

print("Starting instruction fine-tuning (SFT)...")
trainer_stats = trainer.train()
print(f"Training complete. Final loss: {trainer_stats.training_loss:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/85 [00:00<?, ? examples/s]

Starting instruction fine-tuning (SFT)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 85 | Num Epochs = 3 | Total steps = 33
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.158321
20,0.983558
30,0.746666


Training complete. Final loss: 0.9249


## Step 8 — Save SFT adapter

In [17]:
SFT_ADAPTER_PATH = "./outputs/stage2_sft_adapter"
model.save_pretrained(SFT_ADAPTER_PATH)
tokenizer.save_pretrained(SFT_ADAPTER_PATH)
print(f"SFT adapter saved to: {SFT_ADAPTER_PATH}")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    model.save_pretrained("/content/drive/MyDrive/finance_ft/stage2_sft_adapter")
    print("SFT adapter also saved to Google Drive")
except:
    print("Not on Colab — skipping Drive save")

Unsloth: Restored added_tokens_decoder metadata in ./outputs/stage2_sft_adapter/tokenizer_config.json.


SFT adapter saved to: ./outputs/stage2_sft_adapter
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SFT adapter also saved to Google Drive


## Step 9 — Run inference on 10 test questions
> Save these outputs for sft_model_comparison.md

In [18]:
FastLanguageModel.for_inference(model)

test_questions = [
    "What is a mutual fund SIP?",
    "What is the difference between a savings account and a fixed deposit?",
    "How does compound interest work?",
    "What is a credit score and why does it matter?",
    "How can I save tax under Section 80C?",
    "What is the repo rate and how does it affect loans?",
    "What is the difference between term insurance and whole life insurance?",
    "How does the stock market work?",
    "What is a demat account and how do I open one?",
    "What is the difference between a mutual fund and an ETF?",
]

sft_outputs = {}
for q in test_questions:
    prompt = ALPACA_TEMPLATE_INFERENCE.format(q)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = 150,
        temperature    = 0.3,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the answer part
    answer = answer.split("### Answer:")[-1].strip()
    sft_outputs[q] = answer
    print(f"Q: {q}")
    print(f"A: {answer[:200]}")
    print("-" * 60)

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Q: What is a mutual fund SIP?
A: A one-time investment from your personal funds in a mutual fund. The other way to invest in a mutual fund is to invest through a managed investment scheme scheme, which can be done in several ways. A 
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the difference between a savings account and a fixed deposit?
A: A savings account is a financial product which allows you to deposit money and earn interest. A fixed deposit is a financial product which allows you to invest money and earn interest, but the amount 
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What is the difference between term insurance and whole life insurance?
A: The main difference is the kind of insurance. Term insurance is a contract that provides you with a sum amount in exchange for paying premiums. It does not offer the benefits of whole life insurance. 
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How does the stock market work?
A: The price of a stock is determined by supply and demand. When there is more demand than there is supply, the price goes up. When there is more supply than demand, the price goes down.  The price is de
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is a demat account and how do I open one?
A: Dematerialization is a process by which equities and ETFs are physically transferred from a broker to your demat account. There are 2 ways to dematerialize shares - dematerialization order and demater
------------------------------------------------------------
Q: What is the difference between a mutual fund and an ETF?
A: A mutual fund is a company that is directly involved in managing the investments of other people's money.  It's like a company in your own right, but smaller.  It has its own account at the exchange, 
------------------------------------------------------------


##STEP 11- FULL TESTING

In [20]:
from datasets import load_dataset

ds = load_dataset("gbharti/finance-alpaca")

# Filter valid records same way as training
all_records = [
    item for item in ds["train"]
    if len(item["instruction"].strip()) > 10
    and len(item["output"].strip()) > 50
]

# Training used first 100 → test from 500 onwards (safe gap)
test_records = all_records[500:510]  # 10 unseen examples

# Extract just the questions
test_questions = [r["instruction"].strip() for r in test_records]
test_answers   = [r["output"].strip() for r in test_records]  # ground truth

print("Test questions (unseen during training):")
for i, q in enumerate(test_questions):
    print(f"\n{i+1}. {q}")

Test questions (unseen during training):

1. Does wash sale apply if I buy stock on 2 two different dates and sell it later

2. Short Term Capital Gains tax vs. IRA Withdrawal Tax w/o Quarterly Est. Taxes

3. Are there any disadvantages to DHA Investment Properties?

4. Stock Trade Transaction Fee - at what point is it worth it

5. Why would a car company lend me money at a very low interest rate?

6. Swap hedging a currency hedge

7. Under what circumstance will the IRS charge you a late-payment penalty for taxes?

8. Buy car vs lease vs long term rent for 10 years period

9. How do dividend reinvestment purchases work?

10. Are traders 100% responsible for a stock's price changes?


In [22]:
import torch
import gc
from unsloth import FastLanguageModel

# # ── 10 standard test questions ──
# test_questions = [
#     "What is a mutual fund SIP?",
#     "What is the difference between a savings account and a fixed deposit?",
#     "How does compound interest work?",
#     "What is a credit score and why does it matter?",
#     "How can I save tax under Section 80C?",
#     "What is the repo rate and how does it affect loans?",
#     "What is the difference between term insurance and whole life insurance?",
#     "How does the stock market work?",
#     "What is a demat account and how do I open one?",
#     "What is the difference between a mutual fund and an ETF?",
# ]

MAX_SEQ_LEN  = 512
LOAD_IN_4BIT = True

# Instruction format — same for all 3 models
# This is intentional — shows only Stage 2 learned to follow instructions
PROMPT = """Below is a finance question. Write a clear, accurate, and helpful response.

### Question:
{}

### Answer:
"""

def run_test(model, tokenizer, questions, label):
    FastLanguageModel.for_inference(model)
    print("\n" + "=" * 70)
    print(f"  {label}")
    print("=" * 70)

    results = {}
    for q in questions:
        prompt  = PROMPT.format(q)
        inputs  = tokenizer(
            prompt,
            return_tensors="pt"
        ).to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens = 150,
            temperature    = 0.3,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
        generated   = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )
        answer      = generated.split("### Answer:")[-1].strip()
        results[q]  = answer

        print(f"\nQ: {q}")
        print(f"A: {answer[:300]}")
        print("-" * 70)

    return results

def clear_model(model):
    """Delete model and free VRAM"""
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("\nVRAM cleared")


# ════════════════════════════════════════
# MODEL 1 — Base model (no fine-tuning)
# ════════════════════════════════════════
print("Loading base model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Llama-3.2-1B",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)
base_results = run_test(
    model, tokenizer,
    test_questions,
    "MODEL 1 — BASE (no fine-tuning)"
)
clear_model(model)


# ════════════════════════════════════════
# MODEL 2 — Stage 1 non-instruction FT
# ════════════════════════════════════════
print("\nLoading Stage 1 adapter...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "/content/drive/MyDrive/finance_ft/stage1_adapter",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)
stage1_results = run_test(
    model, tokenizer,
    test_questions,
    "MODEL 2 — STAGE 1 (non-instruction fine-tuned)"
)
clear_model(model)


# ════════════════════════════════════════
# MODEL 3 — Stage 2 SFT
# ════════════════════════════════════════
print("\nLoading Stage 2 SFT adapter...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "/content/drive/MyDrive/finance_ft/stage2_sft_adapter",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)
stage2_results = run_test(
    model, tokenizer,
    test_questions,
    "MODEL 3 — STAGE 2 (instruction fine-tuned)"
)
clear_model(model)


# ════════════════════════════════════════
# SIDE BY SIDE COMPARISON
# ════════════════════════════════════════

print("\n" + "=" * 70)
print("COMPARISON WITH GROUND TRUTH")
print("=" * 70)

for i, q in enumerate(test_questions):
    print(f"\n📌 QUESTION: {q}")
    print(f"\n📗 GROUND TRUTH:\n{test_answers[i][:250]}")
    print(f"\n🔵 BASE MODEL:\n{base_results[q][:250]}")
    print(f"\n🟡 STAGE 1:\n{stage1_results[q][:250]}")
    print(f"\n🟢 STAGE 2 SFT:\n{stage2_results[q][:250]}")
    print("=" * 70)

Loading base model...
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-unsloth-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  MODEL 1 — BASE (no fine-tuning)


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Q: Under what circumstance will the IRS charge you a late-payment penalty for taxes?
A: The "real" question is what to do if you miss a payment.  You have 3 choices:  Pay immediately or the cost is zero, file a payment plan with the IRS, or go to court.  Court is expensive and not necessarily successful.  You need good legal counsel.
----------------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Buy car vs lease vs long term rent for 10 years period
A: I'm a big fan of the lease vs buy analysis.  For most people, a car is a big ticket item, and should be considered that way.  However, if you have to decide today about a car purchase, I'd rather have it a lease.  You save on maintenance, and get a no-hassle exchange if you need it.  You also don't 
----------------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: How do dividend reinvestment purchases work?
A: You'd simply place a limit order to buy shares at a specific price, and if it gets filled you'd let that capital grow.  If it doesn't get filled, then you'd let it expire.  You'd also place a limit order to sell shares at a specific price, and if it gets filled you'd let that capital grow.  If it do
----------------------------------------------------------------------

Q: Are traders 100% responsible for a stock's price changes?
A: No.  Traders are not 100% responsible for stock prices.  Instead, stock prices are determined by many factors, many of which are out of the control of the traders.  One of the key factors is the economy, which is out of the control of the traders.  As the economy deteriorates, so do stocks' prices; 
----------------------------------------------------------------------

VRAM cleared

COMPARISON WITH GROUND TRUTH

📌 QUESTION: Does wash sale apply if I buy stock on 2 two different dates and sell it later

📗 

In [23]:
# Install
!pip install rouge_score -q

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def evaluate_model(results, test_questions, test_answers, label):
    print(f"\n{'='*60}")
    print(f"ROUGE SCORES — {label}")
    print(f"{'='*60}")

    r1_scores, r2_scores, rL_scores = [], [], []

    for i, q in enumerate(test_questions):
        prediction  = results[q]
        reference   = test_answers[i]

        scores = scorer.score(reference, prediction)

        r1_scores.append(scores["rouge1"].fmeasure)
        r2_scores.append(scores["rouge2"].fmeasure)
        rL_scores.append(scores["rougeL"].fmeasure)

        print(f"\nQ: {q[:60]}...")
        print(f"ROUGE-1: {scores['rouge1'].fmeasure:.3f} | "
              f"ROUGE-2: {scores['rouge2'].fmeasure:.3f} | "
              f"ROUGE-L: {scores['rougeL'].fmeasure:.3f}")

    # Average scores
    print(f"\n{'─'*60}")
    print(f"AVERAGE SCORES:")
    print(f"ROUGE-1: {sum(r1_scores)/len(r1_scores):.3f}")
    print(f"ROUGE-2: {sum(r2_scores)/len(r2_scores):.3f}")
    print(f"ROUGE-L: {sum(rL_scores)/len(rL_scores):.3f}")

    return {
        "rouge1": sum(r1_scores)/len(r1_scores),
        "rouge2": sum(r2_scores)/len(r2_scores),
        "rougeL": sum(rL_scores)/len(rL_scores),
    }

# Evaluate all 3
base_scores   = evaluate_model(base_results,   test_questions, test_answers, "BASE MODEL")
stage1_scores = evaluate_model(stage1_results, test_questions, test_answers, "STAGE 1 NON-INSTRUCTION")
stage2_scores = evaluate_model(stage2_results, test_questions, test_answers, "STAGE 2 SFT")

# Final comparison table
print("\n\n" + "="*60)
print("FINAL ROUGE COMPARISON")
print("="*60)
print(f"{'Model':<25} {'ROUGE-1':>8} {'ROUGE-2':>8} {'ROUGE-L':>8}")
print("-"*60)
print(f"{'Base model':<25} {base_scores['rouge1']:>8.3f} {base_scores['rouge2']:>8.3f} {base_scores['rougeL']:>8.3f}")
print(f"{'Stage 1 non-instruction':<25} {stage1_scores['rouge1']:>8.3f} {stage1_scores['rouge2']:>8.3f} {stage1_scores['rougeL']:>8.3f}")
print(f"{'Stage 2 SFT':<25} {stage2_scores['rouge1']:>8.3f} {stage2_scores['rouge2']:>8.3f} {stage2_scores['rougeL']:>8.3f}")

  Preparing metadata (setup.py) ... done

ROUGE SCORES — BASE MODEL

Q: Does wash sale apply if I buy stock on 2 two different dates...
ROUGE-1: 0.454 | ROUGE-2: 0.105 | ROUGE-L: 0.309

Q: Short Term Capital Gains tax vs. IRA Withdrawal Tax w/o Quar...
ROUGE-1: 0.198 | ROUGE-2: 0.046 | ROUGE-L: 0.133

Q: Are there any disadvantages to DHA Investment Properties?...
ROUGE-1: 0.177 | ROUGE-2: 0.020 | ROUGE-L: 0.108

Q: Stock Trade Transaction Fee - at what point is it worth it...
ROUGE-1: 0.082 | ROUGE-2: 0.000 | ROUGE-L: 0.082

Q: Why would a car company lend me money at a very low interest...
ROUGE-1: 0.244 | ROUGE-2: 0.040 | ROUGE-L: 0.198

Q: Swap hedging a currency hedge...
ROUGE-1: 0.056 | ROUGE-2: 0.000 | ROUGE-L: 0.056

Q: Under what circumstance will the IRS charge you a late-payme...
ROUGE-1: 0.133 | ROUGE-2: 0.017 | ROUGE-L: 0.133

Q: Buy car vs lease vs long term rent for 10 years period...
ROUGE-1: 0.240 | ROUGE-2: 0.034 | ROUGE-L: 0.164

Q: How do dividend reinvestment purch

In [24]:
!pip install bert_score -q
from bert_score import score as bert_score
import torch

def evaluate_bertscore(results, test_questions, test_answers, label):
    predictions = [results[q] for q in test_questions]
    references  = test_answers

    P, R, F1 = bert_score(
        predictions,
        references,
        lang    = "en",
        device  = "cuda" if torch.cuda.is_available() else "cpu",
        verbose = False
    )

    avg_f1 = F1.mean().item()
    print(f"{label:<30} BERTScore F1 = {avg_f1:.3f}")
    return avg_f1

print("Computing BERTScore...")
base_bert   = evaluate_bertscore(base_results,   test_questions, test_answers, "Base model")
stage1_bert = evaluate_bertscore(stage1_results, test_questions, test_answers, "Stage 1 non-instruction")
stage2_bert = evaluate_bertscore(stage2_results, test_questions, test_answers, "Stage 2 SFT")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.5 MB/s eta 0:00:00
Computing BERTScore...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Stage 1 non-instruction        BERTScore F1 = 0.819


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Stage 2 SFT                    BERTScore F1 = 0.828
